# Convert Qwen3 30B A3B for Triton vLLM

Run this notebook inside the Triton Control code-server workspace. It downloads `Qwen/Qwen3-30B-A3B`, places the model snapshot under `qwen3_30b_a3b/1/model`, and writes the `model.json` file used by Triton's vLLM backend.

The download is large. Use a workspace volume with enough free space before running all cells.

In [ ]:
import json
from pathlib import Path

from huggingface_hub import snapshot_download

In [ ]:
MODEL_ID = "Qwen/Qwen3-30B-A3B"
MODEL_ROOT = Path("qwen3_30b_a3b/1")
MODEL_DIR = MODEL_ROOT / "model"
MODEL_JSON = MODEL_ROOT / "model.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Preparing {MODEL_ID} under {MODEL_DIR}")

In [ ]:
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
    ignore_patterns=[".git*", "*.ckpt", "*.h5", "*.msgpack", "*.onnx"],
)

print(f"Downloaded model snapshot to {MODEL_DIR}")

In [ ]:
engine_config = {
    "model": "./model",
    "tokenizer": "./model",
    "dtype": "auto",
    "max_model_len": 12000,
    "max_num_seqs": 2,
    "max_num_batched_tokens": 4096,
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.7,
    "trust_remote_code": True,
    "enable_prefix_caching": False,
    "enforce_eager": True,
}

MODEL_JSON.write_text(json.dumps(engine_config, indent=2) + "\n", encoding="utf-8")
print(MODEL_JSON.read_text(encoding="utf-8"))

After the notebook finishes, deploy the `qwen3-30b-a3b-vllm` folder with the Triton Control Deploy extension. The extension detects `backend: "vllm"` and uses the vLLM repository sync path automatically.